# Neural Networks for Image Processing - Exercise


**Business problem:** recognize a single scanned digit and decide what needs human review. MNIST is a benchmark,
not a representative sample of operational documents. One observation is one 28 x 28 grayscale image; the target is a digit 0-9.

In this notebook, we will:
1. Prepare image tensors without changing the official test boundary.
2. Compare a raw-pixel linear baseline with a small CNN.
3. Select a pipeline using validation cross-entropy, then evaluate fixed choices.
4. Interpret digit-level errors, review needs and limits of benchmark evidence.

**SW07 Lecture 3. Prerequisites:** neural-network training, classification and validation from Lectures 1 and 2.
One prepared-results activity fits the 10-minute lecture slot; the full sequence is a separate approximately 60-minute practical,
plus first-time downloads and model training. Do not count cold execution inside the lecture's 45 minutes.
The architecture uses two 3 x 3 convolutions (32/64 filters), pooling, dropout 0.5, Adam, batch 128 and five epochs.
Both models use final-epoch weights; validation chooses between pipelines, not between test results.


Read the [concepts](cnn_mnist_concept.ipynb) first. Replace the four `None` values marked with `# TODO` using the instructions above their cells. Those cells are intentionally incomplete. Loading, loops and plotting are supplied. Write short answers at each **Your interpretation** prompt, then check the [solution](cnn_mnist_solution.ipynb).

## Libraries and reproducibility

Install [the pinned TensorFlow environment](../requirements-cnn-mnist.txt) in a separate Python 3.9-3.11 environment:
`pip install -r ../requirements-cnn-mnist.txt` from this notebook's directory.
The kernel must use that environment. Package pins describe the tested stack, not every operating system.

Keras downloads the official `mnist.npz` archive on first use and caches it under `~/.keras/datasets/`.
A later run can use the cache without a dataset download. This is separate from the local Fashion-MNIST files used in Lecture 4.
The cell below fixes random seeds and requests deterministic TensorFlow operations; devices and versions can still affect results.

In [ ]:
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 7
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 5
keras.utils.set_random_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.experimental.enable_op_determinism()
print({name: importlib.metadata.version(name)
       for name in ['tensorflow', 'numpy', 'pandas', 'scikit-learn', 'matplotlib']})

## 1. Prepare identical images and a validation split

Use the official 60,000-image training partition. Split it into 54,000 training and 6,000 validation images with stratification
and seed 7. The loader also returns the 10,000 official test images; keep them raw and unused until Step 5.
Set `pixel_scale` to the maximum value of an 8-bit pixel, so division maps pixel values to [0, 1].
This fixed transformation learns nothing from validation or test images.

The final axis is the single grayscale channel. Targets are one-hot vectors of length ten, matching categorical cross-entropy.
The image grid contains training examples, not examples selected for successful predictions.

In [ ]:
(raw_images, raw_labels), (raw_test_images, raw_test_labels) = keras.datasets.mnist.load_data()
training_indices, validation_indices = train_test_split(
    np.arange(len(raw_labels)), test_size=6000, stratify=raw_labels, random_state=SEED
)
# TODO: Set the maximum value of an 8-bit pixel.
pixel_scale = None
train_images = raw_images[training_indices].astype('float32')[..., None] / pixel_scale
validation_images = raw_images[validation_indices].astype('float32')[..., None] / pixel_scale
train_labels = raw_labels[training_indices]
validation_labels = raw_labels[validation_indices]
train_targets = keras.utils.to_categorical(train_labels, NUM_CLASSES)
validation_targets = keras.utils.to_categorical(validation_labels, NUM_CLASSES)
assert not np.intersect1d(training_indices, validation_indices).size
print('Training / validation / untouched official test:', len(train_images), len(validation_images), len(raw_test_images))
print('Input shape:', train_images.shape[1:], '| target shape:', train_targets.shape[1:])
figure, axes = plt.subplots(1, 10, figsize=(11, 2), layout='constrained')
for digit, axis in enumerate(axes):
    position = np.flatnonzero(train_labels == digit)[0]
    axis.imshow(train_images[position, ..., 0], cmap='gray', vmin=0, vmax=1)
    axis.set_title(str(digit))
    axis.axis('off')
plt.show()

Why must the validation images remain outside fitting, even though division by 255 is fixed?

**Your interpretation:** Replace this sentence with your answer.

## 2. Establish a business and statistical baseline

A training-majority rule always predicts the most frequent training digit. Its probability baseline uses training class proportions.
Do not assume exactly 10% accuracy: the ten digits are not perfectly balanced.
A second baseline is a trainable linear softmax model on the same 784 raw pixels. It tests whether convolutions add value.
Correct routing and reduced review workload are business outcomes; baseline accuracy is only a technical starting point.

In [ ]:
prior_baseline = DummyClassifier(strategy='prior')
prior_baseline.fit(np.zeros((len(train_labels), 1)), train_labels)
prior_validation_probabilities = prior_baseline.predict_proba(np.zeros((len(validation_labels), 1)))
print('Training-majority digit:', int(prior_baseline.classes_[prior_baseline.class_prior_.argmax()]))
print('Prior validation accuracy:', accuracy_score(validation_labels, prior_validation_probabilities.argmax(axis=1)))
print('Prior validation log loss:', log_loss(validation_labels, prior_validation_probabilities, labels=np.arange(NUM_CLASSES)))

What additional evidence does the trainable pixel baseline provide beyond guessing the training-majority digit?

**Your interpretation:** Replace this sentence with your answer.

## 3. Build the two pipelines

Set `kernel_size` to the height/width tuple for a 3 x 3 filter. Both convolutions use it with valid padding.
For the CNN, follow `28 -> 26 -> 13 -> 11 -> 5` in each spatial dimension. Flattening then gives 1,600 activations.
Convolutions learn shared local filters; ReLU introduces nonlinearity; pooling reduces resolution.
Dropout masks half the flattened activations in expectation during training and is inactive in ordinary prediction.

Both pipelines end with ten softmax probabilities. The CNN has 34,826 trainable parameters; the linear baseline has 7,850.
More parameters alone do not guarantee better generalization. The grid below is a benchmark image, not a complete document.

In [ ]:
# TODO: Set the height/width tuple for a 3 x 3 filter.
kernel_size = None
linear_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])
cnn_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=kernel_size, activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=kernel_size, activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])
models = {'Linear softmax': linear_model, 'CNN': cnn_model}
print('Trainable parameters:', {name: model.count_params() for name, model in models.items()})
cnn_model.summary()

What is shared across image positions, and which of the supplied layers have no trainable parameters?

**Your interpretation:** Replace this sentence with your answer.

## 4. Fit with a fixed budget and select on validation data

Set `loss_name` to the Keras categorical cross-entropy loss for one-hot targets and softmax outputs.
Both models use Adam's default learning rate (0.001), batch 128 and five epochs. No test images enter training or selection.
Use final-epoch weights, not a retrospectively chosen test-best checkpoint. Select the pipeline with the **lowest validation log loss**.

Training loss for the CNN is measured with dropout active, while validation uses inference mode.
A lower validation than training loss therefore does not by itself indicate leakage.
Accuracy measures correct class decisions; cross-entropy also evaluates the probability assigned to the true class.

In [ ]:
# TODO: Choose categorical cross-entropy for one-hot targets.
loss_name = None
histories = {}
validation_rows = {}
for name, model in models.items():
    keras.utils.set_random_seed(SEED)
    model.compile(loss=loss_name, optimizer='adam', metrics=['accuracy'])
    histories[name] = model.fit(
        train_images, train_targets, batch_size=BATCH_SIZE, epochs=EPOCHS,
        validation_data=(validation_images, validation_targets), shuffle=True, verbose=2
    )
    measured = model.evaluate(validation_images, validation_targets, verbose=0)
    validation_rows[name] = {'Validation log loss': measured[0], 'Validation accuracy': measured[1]}
validation_results = pd.DataFrame.from_dict(validation_rows, orient='index')
# TODO: Select the row with minimum validation log loss.
selected_model = None
display(validation_results.round(4))
print('Validation-selected model:', selected_model)
figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
for axis, (name, history) in zip(axes, histories.items()):
    epoch_numbers = np.arange(1, len(history.history['loss']) + 1)
    axis.plot(epoch_numbers, history.history['loss'], label='Training')
    axis.plot(epoch_numbers, history.history['val_loss'], label='Validation')
    axis.set(title=name, xlabel='Epoch', ylabel='Categorical cross-entropy')
    axis.legend()
plt.show()

Which pipeline would you select under the declared rule? Why is five epochs not an equal-compute comparison?

**Your interpretation:** Replace this sentence with your answer.

## 5. Evaluate fixed choices on the official test partition

Now apply the same fixed pixel scaling to the official test images. Report accuracy and log loss for the prior rule and both models.
Keep the validation-selected pipeline regardless of test ranking. These are historical benchmark results, not a new deployment guarantee.
Do not use this test set to choose dropout, filters, epoch count or review thresholds.

In [ ]:
test_images = raw_test_images.astype('float32')[..., None] / pixel_scale
test_targets = keras.utils.to_categorical(raw_test_labels, NUM_CLASSES)
test_probabilities = {'Training prior': prior_baseline.predict_proba(np.zeros((len(raw_test_labels), 1)))}
test_probabilities.update({name: model.predict(test_images, batch_size=256, verbose=0)
                           for name, model in models.items()})
test_predictions = {name: probabilities.argmax(axis=1) for name, probabilities in test_probabilities.items()}
test_results = pd.DataFrame.from_dict({name: {
    'Test accuracy': accuracy_score(raw_test_labels, test_predictions[name]),
    'Test log loss': log_loss(raw_test_labels, probabilities, labels=np.arange(NUM_CLASSES)),
} for name, probabilities in test_probabilities.items()}, orient='index')
display(test_results.round(4))
print('Validation-selected model, unchanged:', selected_model)

How does the selected pipeline compare with both baselines? What should happen if the test ranking differs from validation?

**Your interpretation:** Replace this sentence with your answer.

## 6. Interpret errors and business consequences

Rows of the confusion matrix are true digits; columns are predicted digits. Inspect per-digit errors as well as overall accuracy.
The grid shows the first twelve test errors, or the first twelve test examples if there are none. This is explicit error inspection,
not a representative sample of all cases. Images show what happened, not why the model made a causal decision.

In [ ]:
selected_predictions = test_predictions[selected_model]
figure, axis = plt.subplots(figsize=(7, 6), layout='constrained')
ConfusionMatrixDisplay.from_predictions(
    raw_test_labels, selected_predictions, labels=np.arange(NUM_CLASSES),
    values_format='d', colorbar=False, cmap='Greens', ax=axis
)
axis.set_title(selected_model)
plt.show()
error_positions = np.flatnonzero(selected_predictions != raw_test_labels)
positions = error_positions[:12] if len(error_positions) else np.arange(12)
figure, axes = plt.subplots(2, 6, figsize=(11, 4), layout='constrained')
for axis in axes.flat:
    axis.axis('off')
for axis, position in zip(axes.flat, positions):
    axis.imshow(raw_test_images[position], cmap='gray', vmin=0, vmax=255)
    axis.set_title(f'True {raw_test_labels[position]}, predicted {selected_predictions[position]}', fontsize=8)
plt.show()

Identify one observed digit confusion. Why could it matter more than another confusion in a document workflow?

**Your interpretation:** Replace this sentence with your answer.

## Business implications and limitations

MNIST digits are cropped, centered and grayscale; actual documents contain multiple characters, noise and changing capture devices.
This comparison uses one random seed and a fixed five-epoch budget. It does not optimize either architecture or isolate every source of gain.
Validation is development evidence. Repeated test inspection is not fresh confirmation.
Probability calibration, unfamiliar inputs, human review, latency, privacy and monitoring need separate evaluation.
Associations learned from pixels do not establish causal effects on business outcomes.

Would you automatically process documents with high maximum softmax probability? Specify the next evidence you need.

**Your interpretation:** Replace this sentence with your answer.

## Summary and optional exercises

- Convolutions learn shared local patterns; flattening and a linear model provide a useful comparison.
- Train on training images, select on validation and evaluate fixed choices on the official test partition.
- Benchmark accuracy, trustworthy probabilities and business value are different claims.

**Optional practical:** predeclare one pooling or dropout change and compare using training/validation only.
**Further exercise:** test a controlled brightness or rotation shift on validation images. Do not treat this as a representative production shift.
Obtain fresh evaluation data before improving the system in response to historical test errors.

## References

**Required student reading:** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023).
*An Introduction to Statistical Learning: With Applications in Python*. Springer. Neural-network material.
https://www.statlearning.com/

**Dataset:** LeCun, Y., Cortes, C., & Burges, C. J. C. *The MNIST Database of Handwritten Digits*.
https://yann.lecun.com/exdb/mnist/

**Official technical documentation:** [Keras MNIST](https://keras.io/api/datasets/mnist/),
[Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/),
[MaxPooling2D](https://keras.io/api/layers/pooling_layers/max_pooling2d/),
[Dropout](https://keras.io/api/layers/regularization_layers/dropout/).

**Optional reading:** Zhang, A., Lipton, Z. C., Li, M., & Smola, A. J.
*Dive into Deep Learning*. Convolutional neural networks. https://d2l.ai/

**Environment:** [Pinned CNN requirements](../requirements-cnn-mnist.txt).
Practical notebooks require TensorFlow; the small calculations here use NumPy, pandas and matplotlib.

### Jupyter notebook - footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime

print(platform.platform())
print('Python:', platform.python_version())
print('Date:', datetime.now().isoformat(timespec='seconds'))